In [1]:
import numpy as np
import pandas as pd
import zarr
import torch
import math
from datetime import date, datetime, timedelta
import statsmodels.api as sm
import os
import gc
import re
import shutil
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
INPUT_DIR = "/data_3/scratch/francesco/new_zarr.zarr"
ds = xr.open_zarr(INPUT_DIR, chunks="auto")
ds

/home/francesco/miniconda3/envs/ndvi/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:99: UserWarning: The codec `vlen-bytes` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/home/francesco/miniconda3/envs/ndvi/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


<xarray.Dataset> Size: 12GB
Dimensions:        (last_date_idx: 8, pixel: 1000014, param: 6, date: 3073)
Coordinates:
  * pixel          (pixel) int64 8MB 85668616 85668617 ... 94882985 94882986
  * param          (param) object 48B 'par0' 'par1' 'par2' 'par3' 'par4' 'par5'
  * last_date_idx  (last_date_idx) int64 64B 0 1 2 3 4 5 6 7
  * date           (date) datetime64[ns] 25kB 2017-04-03 ... 2025-08-31
Data variables:
    last_dates     (last_date_idx, pixel) object 64MB dask.array<chunksize=(8, 100), meta=np.ndarray>
    params_upper   (pixel, param) float32 24MB dask.array<chunksize=(100, 6), meta=np.ndarray>
    median_ndvi    (pixel, date) int16 6GB dask.array<chunksize=(100, 365), meta=np.ndarray>
    counter        (date) int16 6kB dask.array<chunksize=(365,), meta=np.ndarray>
    params_lower   (pixel, param) float32 24MB dask.array<chunksize=(100, 6), meta=np.ndarray>
    ndvi           (pixel, date) int16 6GB dask.array<chunksize=(100, 365), meta=np.ndarray>

In [3]:
def find_min_max_dates_chunk(last_dates_array,first_date,current_date):

    
    start_idx = int((last_dates_array.min() - first_date)  / np.timedelta64(1, 'D'))
    end_idx = int((current_date - first_date)  / np.timedelta64(1, 'D'))

    start_idx = max(0,start_idx -1)

    return start_idx, end_idx


In [4]:

def smoothing_and_gapfilling(ndvi_arr,median_ndvi_arr,last_array_dates_idx,last_delta,current_delta,deltas_arr, pot_outlier_present):

    if pot_outlier_present:

        pot_date_idx = last_array_dates_idx[7]

        pot_ndvi = ndvi_arr[pot_date_idx] / 10000
        pot_median_ndvi = median_ndvi_arr[pot_date_idx] / 10000

        pot_delta = pot_ndvi - pot_median_ndvi

        # perform L1 linear gapfilling

        idx_to_interpolate = np.arange(last_array_dates_idx[6], len(ndvi_arr))
        deltas_L1 = np.array([last_delta, pot_delta,current_delta])
        deltas_L1_idx = np.array([last_array_dates_idx[6],last_array_dates_idx[7], len(ndvi_arr)])

        deltas_interpolated = np.interp(idx_to_interpolate,deltas_L1_idx,deltas_L1)

        L1_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[6]: ] / 10000

        ndvi_arr[last_array_dates_idx[6]: ] = L1_ndvi

        # perform L2 smoothing

        idx_to_interpolate = np.arange(last_array_dates_idx[2], last_array_dates_idx[4] +1)

        deltas_L2 = np.concatenate(deltas_arr,np.array([current_delta]))
        deltas_L2_idx = last_array_dates_idx[2:4]

        smoothed_deltas =  sm.nonparametric.lowess(deltas_L2, np.arange(1,9), frac= 1, it=3, return_sorted=False)
        deltas_interpolated = np.interp(idx_to_interpolate,deltas_L2_idx,smoothed_deltas)


        L2_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[4]] / 10000

        ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[4]] = L2_ndvi

    else:
        
        # perform L1 linear gapfilling

        idx_to_interpolate = np.arange(last_array_dates_idx[6], len(ndvi_arr))
        deltas_interpolated = np.linspace(last_delta, current_delta,num = len(idx_to_interpolate))

        L1_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[6]: ] / 10000

        ndvi_arr[last_array_dates_idx[6]: ] = L1_ndvi

        # perform L2 smoothing
        idx_to_interpolate = np.arange(last_array_dates_idx[2], last_array_dates_idx[3] +1)

        smoothed_deltas =  sm.nonparametric.lowess(deltas_arr, np.arange(1,8), frac= 1, it=3, return_sorted=False)
        deltas_interpolated = np.linspace(smoothed_deltas[2], smoothed_deltas[3],num = len(idx_to_interpolate))

        L2_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[3] +1] / 10000

        ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[3] +1] = L2_ndvi
    
    return ndvi_arr

def update_date(last_dates_array,current_date,pot_outlier_present):

    if pot_outlier_present:

        new_last_dates_array = np.concatenate([last_dates_array[2:6],
                                               np.array([
                                                    last_dates_array[7],
                                                    last_dates_array[6],
                                                    current_date,
                                                    np.datetime64("1900-01-01", "D")
                                                    ])]).astype("datetime64[D]")



        """[last_dates_array[2:6], 
                                               np.array([
                                                    last_dates_array[7],
                                                    last_dates_array[6],
                                                    current_date,
                                                    np.datetime64("1900-01-01", "D")
                                                    ])]"""
    
    else:

        new_last_dates_array = np.concatenate([last_dates_array[1:7],
                                               np.array([
                                                    current_date,
                                                    np.datetime64("1900-01-01", "D")])
                                               ]).astype("datetime64[D]")

    return new_last_dates_array

In [5]:

def continous_ndvi(pixel,last_dates_array,dates,start_date,current_date):

    # load all the data
    current_date = current_date # just for pass it in update_date function
    load_interval = pd.date_range(start_date, current_date, freq="D")
    ndvi_arr = ds["ndvi"].sel(pixel=pixel, date = load_interval).load().values
    median_ndvi_arr = ds["median_ndvi"].sel(pixel=pixel, date = load_interval).load().values

    last_array_dates_idx = (last_dates_array - start_date).astype(int)

    current_ndvi = ndvi_arr[-1] / 10000
    median_current_value = median_ndvi_arr[-1] / 10000

    last_date_idx = last_array_dates_idx[6]

    last_ndvi = ndvi_arr[last_date_idx] / 10000
    last_median_ndvi = median_ndvi_arr[last_date_idx] / 10000
    
    last_delta = last_ndvi - last_median_ndvi
    current_delta = current_ndvi - median_current_value
    delta_delta = current_delta - last_delta

    #print(current_date, current_ndvi)

    if (current_ndvi > 0) & (current_ndvi < 1):

        # check if its a potential outlier or not
        if (abs(delta_delta) > 0.1) & (abs(current_delta) > 0.1):

            # potential outlier, do not do anyhting and return the original array
            last_dates_array[7] = current_delta
            return ndvi_arr, last_dates_array
        
        else:

            deltas_arr = (ndvi_arr[last_array_dates_idx[:7]] - ndvi_arr[last_array_dates_idx[:7]]) / 10000

            # true obs, check if a potential outlier is pending
            if last_array_dates_idx[7] > 0:

                # potential outlier is present
                pot_date_idx = last_array_dates_idx[7]

                pot_ndvi = ndvi_arr[pot_date_idx] / 10000
                pot_median_ndvi = median_ndvi_arr[pot_date_idx] / 10000

                pot_delta = pot_ndvi - pot_median_ndvi

                if abs(pot_delta) < 0.1:

                    pot_deltas_arr = (ndvi_arr[last_array_dates_idx] - ndvi_arr[last_array_dates_idx]) / 10000

                    # the pot. outlier was a true value
                    ndvi_arr = smoothing_and_gapfilling(ndvi_arr,median_ndvi_arr,last_array_dates_idx,last_delta,current_delta,pot_deltas_arr, pot_outlier_present = True)
                    
                    # clear any pot. out. pending
                    last_dates_array = update_date(last_dates_array, current_date, pot_outlier_present = True)

                    return ndvi_arr, last_dates_array

                else:

                    # the value was a pot. outlier, ignored
                    ndvi_arr = smoothing_and_gapfilling(ndvi_arr,median_ndvi_arr,last_array_dates_idx,last_delta,current_delta,deltas_arr, pot_outlier_present = False)
                    
                    # clear any pot. out. pending
                    last_dates_array = update_date(last_dates_array, current_date, pot_outlier_present = False)

                    return ndvi_arr, last_dates_array

            else:

                # no pot. outlier
                ndvi_arr = smoothing_and_gapfilling(ndvi_arr,median_ndvi_arr,last_array_dates_idx,last_delta,current_delta,deltas_arr, pot_outlier_present = False)
                
                # clear any pot. out. pending
                last_dates_array = update_date(last_dates_array, current_date, pot_outlier_present = False)

                return ndvi_arr, last_dates_array

    else:
             
        # no obs, estimate the current ndvi value
        tau = len(ndvi_arr) - last_date_idx
        estimated_delta = last_delta * np.exp((-tau/45))
        ndvi_arr[-1] = estimated_delta + median_current_value

        return ndvi_arr, last_dates_array
    
    

In [6]:
spinup = 750
pixel = 85668616

ndvi_subset = ds["ndvi"].sel(pixel=pixel)[:spinup].load()

valid_mask = (ndvi_subset > 0) & (ndvi_subset < 10000)

ndvi_subset_t = ndvi_subset[valid_mask][-7:]["date"].values.astype("datetime64[D]")

full_dates = np.concatenate([ndvi_subset_t,np.array([np.datetime64("1900-01-01", "D")])])



In [7]:
# set initial condition
first_date = ds["date"][0].load().astype("datetime64[D]").values
dates = ds["date"].load().astype("datetime64[D]").values
first_date = np.datetime64(first_date, "D")
last_dates_array = full_dates

# run the continous ingestion
for i in np.arange(1,1000):

    current_date = ds["date"][spinup +i].load().astype("datetime64[D]").values
    start_date = last_dates_array[:6].min()
    #last_dates_array = subset["last_dates"].astype("datetime64[D]").values

    print(last_dates_array)
    ndvi_arr, last_dates_array = continous_ndvi(pixel,last_dates_array,dates,start_date,current_date)
    #print(last_dates_array[1])


['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-31' '2018-08-03' '2018-08-05'
 '2018-08-18' '2018-08-20' '1900-01-01']
['2018-07-26' '2018-07-29' '2018-07-3